# Connectome — sprint 15A: **entrenar donde la métrica mira**

**Inputs:** train part1/part2, valid, `Connectome models`. GPU T4 x2. **Save & Run All.** ~2 h.

**El problema:** la métrica solo puntúa el **9 %** de las filas (las marcadas con `is_scored`), vienen en rachas cortas, y esa columna **no existe en los datos de entrenamiento**. Veníamos entrenando parejo sobre las 20.000 filas de cada secuencia, o sea gastando la mayor parte del esfuerzo en filas que no cuentan.

**La solución:** ajustamos un modelo que adivina qué filas van a ser puntuadas, usando solo las 112 features. Da **AUC 0.794**, y la separación es fuerte: el decil con mayor probabilidad tiene **34,9 %** de filas puntuadas contra **0,5 %** del decil más bajo. Con eso pesamos cada fila al entrenar.

| | |
|---|---|
| modelo | LSTM 128x2, lr 5e-4, 10 épocas, SWA 7–10, destilada, semilla 2151 |
| cambio | peso de cada fila = \|y\| × (probabilidad de ser puntuada / promedio) |
| comparar contra | **la alumna LR = 0.6834**: misma receta exacta sin pesar |

Es lo último barato que quedaba sin probar, y es el único cambio que alinea lo que optimizamos con lo que nos puntúan.

In [ ]:
import os, torch
assert torch.cuda.is_available(), 'Activá GPU (Session options -> Accelerator -> GPU T4 x2)'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
for p in ['/kaggle/input/datasets/joaquinfiorina/connectome-train-part1/train_part1.parquet', '/kaggle/input/datasets/joaquinfiorina/connectome-train-part2/train_part2.parquet'] + ['/kaggle/input/datasets/joaquinfiorina/connectome-valid/valid.parquet']:
    assert os.path.isfile(p), f'Falta input: {p}'
for f in ['gru128_distill_seed2029_swa.pt', 'lstm128x2_l1_seed2047_swa.pt', 'lstm128x2_studentLR_seed2072_swa.pt', 'lstm128x2_studentLA_best.pt']:
    assert os.path.isfile(f'/kaggle/input/datasets/joaquinfiorina/connectome-models/{f}'), f'Falta {f}: agregá el dataset Connectome models'
os.makedirs('/kaggle/working/results', exist_ok=True); print('Inputs OK')


In [ ]:
%%writefile utils.py
"""Competition callback and clipped block-wise Weighted Pearson scorer."""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq

SEQUENCE_LENGTH = 20_000
WARMUP = 99
N_FEATURES = 112
FEATURE_COLUMNS = [f'i{i}_{g}{j}' for i in range(2)
                   for g, n in [('p', 22), ('v', 22), ('dp', 4), ('dv', 4)] for j in range(n)]
FEATURE_COLUMNS += [f'a{i}' for i in range(8)]
TARGET_COLUMNS = ('t0', 't1')
METRIC_CLIP = 2.0


@dataclass(frozen=True)
class DataPoint:
    seq_ix: int
    step_in_seq: int
    need_prediction: bool
    state: np.ndarray


def weighted_pearson(target, prediction):
    target = np.asarray(target, dtype=np.float32)
    prediction = np.asarray(prediction, dtype=np.float32)
    if target.shape != prediction.shape or target.ndim != 1:
        raise ValueError('expected equal one-dimensional target and prediction arrays')
    if not np.isfinite(target).all() or not np.isfinite(prediction).all():
        raise ValueError('target/prediction contains nonfinite values')
    # The metric clips both arrays. Stored targets and submitted predictions
    # remain untouched outside this calculation.
    target = np.clip(target, -METRIC_CLIP, METRIC_CLIP).astype(np.float64)
    prediction = np.clip(prediction, -METRIC_CLIP, METRIC_CLIP).astype(np.float64)
    weights = np.abs(target)
    total = weights.sum()
    if total < 1e-8:
        return 0.0
    target_mean = np.sum(weights * target) / total
    prediction_mean = np.sum(weights * prediction) / total
    target_centered = target - target_mean
    prediction_centered = prediction - prediction_mean
    covariance = np.sum(weights * target_centered * prediction_centered) / total
    target_std = np.sqrt(np.sum(weights * target_centered**2) / total)
    prediction_std = np.sqrt(np.sum(weights * prediction_centered**2) / total)
    if target_std <= 1e-8 or prediction_std <= 1e-8:
        return 0.0
    return float(np.clip(covariance / (target_std * prediction_std), -1.0, 1.0))


class BlockAccumulator:
    def __init__(self):
        self.scores = []
        self.blocks_seen = 0
        self.selected_rows = 0
        self.effective_rows = 0

    def add(self, targets, predictions, mask):
        targets = np.asarray(targets, dtype=np.float32)
        predictions = np.asarray(predictions, dtype=np.float32)
        mask = np.asarray(mask, dtype=bool)
        if targets.shape != (SEQUENCE_LENGTH, 2) or predictions.shape != targets.shape or mask.shape != (SEQUENCE_LENGTH,):
            raise ValueError('one accumulator call must contain exactly one complete 20k block')
        if np.any(mask[:WARMUP]):
            raise ValueError('warm-up rows cannot be scored')
        if not np.isfinite(targets).all() or not np.isfinite(predictions[WARMUP:]).all():
            raise ValueError('nonfinite labels or required predictions')
        selected_targets = targets[mask]
        selected_predictions = predictions[mask]
        eligible = all(np.any(selected_targets[:, j] > 0) and
                       np.any(selected_targets[:, j] < 0) for j in range(2))
        self.blocks_seen += 1
        self.selected_rows += int(mask.sum())
        if eligible:
            values = [weighted_pearson(selected_targets[:, j], selected_predictions[:, j])
                      for j in range(2)]
            self.scores.append(values)
            self.effective_rows += int(mask.sum())

    def result(self):
        if not self.scores:
            raise ValueError('no eligible blocks')
        per_side = np.mean(np.asarray(self.scores, dtype=np.float64), axis=0)
        return {'t0': float(per_side[0]), 't1': float(per_side[1]),
                'weighted_pearson': float(per_side.mean()), 'blocks': self.blocks_seen,
                'eligible_blocks': len(self.scores), 'excluded_blocks': self.blocks_seen-len(self.scores),
                'selected_rows': self.selected_rows, 'effective_scored_rows': self.effective_rows,
                'metric_clip': [-METRIC_CLIP, METRIC_CLIP],
                'aggregation': 'equal mean of eligible block-wise two-target clipped Weighted Pearson'}


def validate_sequence(table):
    if table.num_rows != SEQUENCE_LENGTH:
        raise ValueError('each row group must be exactly one 20k sequence')
    ids = table['seq_ix'].to_numpy()
    if not np.all(ids == ids[0]):
        raise ValueError('multiple sequence IDs in one row group')
    if not np.array_equal(table['step_in_seq'].to_numpy(), np.arange(SEQUENCE_LENGTH)):
        raise ValueError('incorrect step order')
    need = table['need_prediction'].to_numpy().astype(bool)
    if not np.array_equal(need, np.arange(SEQUENCE_LENGTH) >= WARMUP):
        raise ValueError('incorrect warm-up contract')
    return int(ids[0]), need


class ScorerStepByStep:
    def __init__(self, dataset_path: str | Path):
        self.parquet = pq.ParquetFile(dataset_path)
        expected = ['seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', *FEATURE_COLUMNS, *TARGET_COLUMNS]
        if self.parquet.schema_arrow.names != expected:
            raise ValueError('local scoring requires the labeled validation schema')

    def score(self, model):
        accumulator = BlockAccumulator()
        seen = set()
        for group in range(self.parquet.num_row_groups):
            table = self.parquet.read_row_group(group, use_threads=False)
            seq, need = validate_sequence(table)
            if seq in seen:
                raise ValueError('duplicate sequence ID')
            seen.add(seq)
            features = np.column_stack([table[c].to_numpy() for c in FEATURE_COLUMNS]).astype(np.float32)
            targets = np.column_stack([table[c].to_numpy() for c in TARGET_COLUMNS]).astype(np.float32)
            predictions = np.full((SEQUENCE_LENGTH, 2), np.nan, dtype=np.float32)
            for step in range(SEQUENCE_LENGTH):
                point = DataPoint(seq, step, bool(need[step]), features[step])
                value = model.predict(point)
                if not need[step]:
                    if value is not None:
                        raise ValueError('return None during warm-up AFTER updating state')
                else:
                    value = np.asarray(value, dtype=np.float32)
                    if value.shape != (2,) or not np.isfinite(value).all():
                        raise ValueError('return two finite scores on every required row')
                    predictions[step] = value
            accumulator.add(targets, predictions, table['is_scored'].to_numpy().astype(bool) & need)
        result = accumulator.result()
        result['rows_seen'] = len(seen)*SEQUENCE_LENGTH
        result['predictions_required'] = len(seen)*(SEQUENCE_LENGTH-WARMUP)
        return result


In [ ]:
%%writefile train_official_gpu_v1.py
r"""v4 (2026-09-18): --distill-cache precalcula los soft targets de los teachers una vez (fp16) en vez de correrlos en cada chunk.
Entrenamiento oficial GPU/CPU (v2): GRU/LSTM por batches, métrica GLOBAL, dropout/weight decay, SWA.

Mismo script para la laptop/PC (CPU) y para Kaggle (GPU): detecta `cuda` solo.

Qué hace distinto a los FirstTry:
  * Entrena B secuencias en paralelo (batches) con BPTT truncado por chunks.
  * Loss alineada a la métrica (targets clip ±2, peso |y|), o -WeightedPearson (`--loss wp`).
  * Gradient clipping, lr configurable, checkpoints por época y `--resume`.
  * Evalúa con la métrica oficial (BlockAccumulator de utils.py) en grupos de valid fijos:
    con `--valid-groups 10 --seed 2026` son los MISMOS 10 grupos que EXPERIMENTS.md
    (baseline = 0.568 ahí).
  * Opcional: lags/deltas de `model/feature_select/features.py` (`--top-k`), flags de centinela.

Uso local (desde challenge/):
    python model/official_gpu/train_official_gpu_v1.py --train-groups 500 --epochs 3
Uso Kaggle:
    python train_official_gpu_v1.py --train-parquet /kaggle/input/connectome-train-part1/train_part1.parquet \
        /kaggle/input/connectome-train-part2/train_part2.parquet \
        --valid-parquet /kaggle/input/connectome-valid/valid.parquet --train-groups all --epochs 5 \
        --output-dir /kaggle/working/results
"""
from __future__ import annotations

import argparse
import csv
import json
import math
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch
from torch import nn

HERE = Path(__file__).resolve().parent
ROOT = HERE.parents[2] if len(HERE.parents) > 2 else HERE   # starter pack root en local; en Kaggle, la misma carpeta
for p in (ROOT, HERE, HERE.parent / "feature_select"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from utils import (BlockAccumulator, FEATURE_COLUMNS, SEQUENCE_LENGTH, TARGET_COLUMNS,  # noqa: E402
                   WARMUP, validate_sequence, weighted_pearson)

SENTINEL = -5.1993
SENTINEL_COLS = ("a5", "a6", "a7")

_IX = {c: i for i, c in enumerate(FEATURE_COLUMNS)}
BIDP = [_IX[f"i0_p{j}"] for j in range(11)]        # 11 niveles de compra (precio)
ASKP = [_IX[f"i0_p{j}"] for j in range(11, 22)]    # 11 niveles de venta
BIDV = [_IX[f"i0_v{j}"] for j in range(11)]
ASKV = [_IX[f"i0_v{j}"] for j in range(11, 22)]
BIDP1 = [_IX[f"i1_p{j}"] for j in range(11)]
ASKP1 = [_IX[f"i1_p{j}"] for j in range(11, 22)]
BIDV1 = [_IX[f"i1_v{j}"] for j in range(11)]
ASKV1 = [_IX[f"i1_v{j}"] for j in range(11, 22)]
CROSS_NAMES = ["deseq_top", "deseq_3", "deseq_total", "deseq_vol_frac", "desvio_bid_p", "desvio_ask_p",
               "desvio_bid_v", "desvio_ask_v", "prof_bid_rel", "prof_ask_rel", "spread_rel", "mid_dif_rel",
               "max_bid_v", "max_ask_v", "conc_bid_v", "conc_ask_v", "deseq_i1_top", "deseq_dif",
               "spread_x_deseq", "deseq_top_x_3"]


def cross_features(x: np.ndarray) -> np.ndarray:
    """(T, 112) -> (T, 20): combinaciones NO LINEALES dentro de la misma fila.

    Solo cosas que la primera capa de la red no puede construir sola: cocientes (desequilibrios),
    desvíos y máximos entre niveles, y productos. Denominadores con +1.0 para que no exploten
    (las entradas ya vienen normalizadas, valores del orden de 1). Causales: solo usan la fila actual.
    """
    bp, ap_, bv, av = x[:, BIDP], x[:, ASKP], x[:, BIDV], x[:, ASKV]
    bv1, av1 = x[:, BIDV1], x[:, ASKV1]
    spread = ap_[:, 0] - bp[:, 0]
    mid = (ap_[:, 0] + bp[:, 0]) * 0.5
    mid1 = (x[:, ASKP1][:, 0] + x[:, BIDP1][:, 0]) * 0.5
    d_top = (bv[:, 0] - av[:, 0]) / (np.abs(bv[:, 0]) + np.abs(av[:, 0]) + 1.0)
    d_3 = (bv[:, :3].sum(1) - av[:, :3].sum(1)) / (np.abs(bv[:, :3]).sum(1) + np.abs(av[:, :3]).sum(1) + 1.0)
    d_all = (bv.sum(1) - av.sum(1)) / (np.abs(bv).sum(1) + np.abs(av).sum(1) + 1.0)
    d_i1 = (bv1[:, 0] - av1[:, 0]) / (np.abs(bv1[:, 0]) + np.abs(av1[:, 0]) + 1.0)
    out = [
        d_top, d_3, d_all,
        np.abs(bv).sum(1) / (np.abs(bv).sum(1) + np.abs(av).sum(1) + 1.0),
        bp.std(1), ap_.std(1), bv.std(1), av.std(1),
        (bp[:, 0] - bp[:, 10]) / (np.abs(spread) + 1.0),
        (ap_[:, 10] - ap_[:, 0]) / (np.abs(spread) + 1.0),
        spread / (np.abs(mid) + 1.0),
        (mid - mid1) / (np.abs(spread) + 1.0),
        bv.max(1), av.max(1),
        bv.max(1) / (np.abs(bv).sum(1) + 1.0),        # concentración: cuánto del volumen está en un solo nivel
        av.max(1) / (np.abs(av).sum(1) + 1.0),
        d_i1, d_top - d_i1,
        spread * d_top, d_top * d_3,
    ]
    return np.clip(np.stack(out, axis=1), -10.0, 10.0).astype(np.float32)


try:
    from scipy.signal import lfilter as _lfilter     # importar UNA vez: adentro de la función cuesta 2,5 s la primera
except ImportError:
    _lfilter = None

EMA_COLS = ["i0_p0", "i0_p11", "i0_v0", "i0_v11", "i1_p0", "i1_p11", "i1_v0", "i1_v11", "a0", "a1", "a2", "a4"]
EMA_IX = [_IX[c] for c in EMA_COLS]
EMA_SPANS = (100, 1000, 10000)      # filas de memoria de cada promedio
EMA_VOL_COLS = ["i0_p0", "i0_p11", "i0_v0", "i1_p0"]
EMA_VOL_IX = [_IX[c] for c in EMA_VOL_COLS]
EMA_NAMES = ([f"{c}_dif_ema{s}" for s in EMA_SPANS for c in EMA_COLS] +
             [f"{c}_vol_lenta" for c in EMA_VOL_COLS])


def _ema(x: np.ndarray, span: int) -> np.ndarray:
    """Promedio móvil exponencial causal por columna (filtro IIR de primer orden). ema[0] = x[0]."""
    a = 2.0 / (span + 1.0)
    if _lfilter is not None:
        zi = (x[0] * (1 - a))[None, :]                      # estado inicial: arranca en x[0]
        return _lfilter([a], [1.0, -(1 - a)], x, axis=0, zi=zi)[0]
    else:                                                    # sin scipy: bucle (lento, solo para pruebas)
        out = np.empty_like(x)
        m = x[0].copy()
        for t in range(len(x)):
            m += a * (x[t] - m)
            out[t] = m
        return out


def ema_features(x: np.ndarray) -> np.ndarray:
    """(T, 112) -> (T, len(EMA_NAMES)): desvío respecto de promedios móviles largos + volatilidad lenta."""
    sub = x[:, EMA_IX].astype(np.float32)
    parts = []
    e100 = None
    for span in EMA_SPANS:
        e = _ema(sub, span).astype(np.float32)
        if span == EMA_SPANS[0]:
            e100 = e
        parts.append(sub - e)
    vol = _ema(np.abs(sub[:, [EMA_COLS.index(c) for c in EMA_VOL_COLS]] -
                      e100[:, [EMA_COLS.index(c) for c in EMA_VOL_COLS]]), 1000).astype(np.float32)
    parts.append(vol)
    return np.clip(np.concatenate(parts, axis=1), -10.0, 10.0).astype(np.float32)


# ----------------------------------------------------------------------------- datos
class ParquetGroups:
    """Varios parquet vistos como una sola lista de row groups (1 row group = 1 secuencia)."""

    def __init__(self, paths: list[str]):
        self.files = [pq.ParquetFile(p) for p in paths]
        self.index = [(f, g) for f in self.files for g in range(f.num_row_groups)]
        self.has_scored = all("is_scored" in f.schema_arrow.names for f in self.files)

    def __len__(self):
        return len(self.index)

    def read(self, i: int):
        f, g = self.index[i]
        cols = ["seq_ix", "step_in_seq", "need_prediction", *FEATURE_COLUMNS, *TARGET_COLUMNS]
        if self.has_scored:
            cols.append("is_scored")
        t = f.read_row_group(g, columns=cols, use_threads=False)
        seq_ix, need = validate_sequence(t)
        x = np.column_stack([t[c].to_numpy() for c in FEATURE_COLUMNS]).astype(np.float32)
        y = np.column_stack([t[c].to_numpy() for c in TARGET_COLUMNS]).astype(np.float32)
        x[~np.isfinite(x)] = 0.0
        scored = t["is_scored"].to_numpy().astype(bool) if self.has_scored else None
        return seq_ix, need, x, y, scored


def select_groups(total: int, count: int, seed: int) -> list[int]:
    """Idéntico a first_try_common.select_groups (mismos grupos que EXPERIMENTS.md)."""
    if not 1 <= count <= total:
        raise ValueError(f"count debe estar entre 1 y {total}")
    return sorted(np.random.default_rng(seed).choice(total, count, replace=False).tolist())


class FeaturePipeline:
    """112 crudas (+ flags de centinela) (+ lags/deltas de features.py). Guarda su normalizador."""

    def __init__(self, top_k: int, sentinel_flags: bool, ranking: Path | None, cross: bool = False, ema: bool = False):
        self.top_k = top_k
        self.sentinel_flags = sentinel_flags
        self.cross = cross
        self.ema = ema
        self.specs = []
        if top_k > 0:
            from features import load_rankings, select_temporal_features  # model/feature_select
            self.specs = select_temporal_features(load_rankings(ranking), top_k)
        self.sent_idx = [FEATURE_COLUMNS.index(c) for c in SENTINEL_COLS]
        self.mean = None
        self.std = None

    @property
    def names(self) -> list[str]:
        out = list(FEATURE_COLUMNS)
        if self.sentinel_flags:
            out += [f"{c}_is_sentinel" for c in SENTINEL_COLS]
        if self.cross:
            out += CROSS_NAMES
        if self.ema:
            out += EMA_NAMES
        out += [s.name for s in self.specs]
        return out

    def raw_to_features(self, x: np.ndarray) -> np.ndarray:
        parts = [x]
        if self.sentinel_flags:
            flags = np.isclose(x[:, self.sent_idx], SENTINEL, atol=2e-4).astype(np.float32)
            x = x.copy()
            x[:, self.sent_idx] = np.where(flags > 0, 0.0, x[:, self.sent_idx])
            parts = [x, flags]
        if self.cross:
            parts.append(cross_features(x))
        if self.ema:
            parts.append(ema_features(x))
        if self.specs:
            from features import build_sequence
            derived = build_sequence(x, self.specs)[:, len(FEATURE_COLUMNS):]
            parts.append(derived)
        return np.concatenate(parts, axis=1) if len(parts) > 1 else parts[0]

    def fit_normalizer(self, groups: ParquetGroups, ids: list[int], rows_per_group: int, seed: int):
        rng = np.random.default_rng(seed)
        sample = []
        for i in ids:
            _, need, x, _, _ = groups.read(i)
            f = self.raw_to_features(x)[need]
            take = min(rows_per_group, len(f))
            sample.append(f[rng.choice(len(f), take, replace=False)])
        s = np.concatenate(sample)
        self.mean = np.nanmean(s, axis=0).astype(np.float32)
        self.std = np.nanstd(s, axis=0).astype(np.float32)
        self.std[self.std < 1e-6] = 1.0
        if self.sentinel_flags:   # flags binarias: no estandarizar
            a, b = len(FEATURE_COLUMNS), len(FEATURE_COLUMNS) + len(SENTINEL_COLS)
            self.mean[a:b] = 0.0
            self.std[a:b] = 1.0

    def transform(self, x: np.ndarray) -> np.ndarray:
        f = (self.raw_to_features(x) - self.mean) / self.std
        return np.nan_to_num(f, nan=0.0).astype(np.float32)   # sin historia -> 0 estandarizado

    def state(self) -> dict:
        return {"top_k": self.top_k, "sentinel_flags": self.sentinel_flags, "cross": self.cross, "ema": self.ema,
                "specs": [(s.base_feature, s.transform, s.lag) for s in self.specs],
                "mean": self.mean, "std": self.std, "names": self.names}



# ----------------------------------------------------------------------------- métrica GLOBAL (nueva, 2026-09)
class GlobalWP:
    """Weighted Pearson global: junta TODAS las filas scoreadas de todas las secuencias y calcula
    una sola correlación por target (clip ±2, peso |y|), luego promedia t0 y t1.
    Reemplaza a la métrica por secuencia (utils.BlockAccumulator), que quedó obsoleta."""

    def __init__(self):
        self.s = np.zeros((2, 6), dtype=np.float64)   # por target: sum w, wy, wp, wyy, wpp, wyp

    def add(self, targets, predictions, mask):
        y = np.clip(targets[mask].astype(np.float64), -2.0, 2.0)
        p = np.clip(predictions[mask].astype(np.float64), -2.0, 2.0)
        w = np.abs(y)
        for j in range(2):
            wj, yj, pj = w[:, j], y[:, j], p[:, j]
            self.s[j] += [wj.sum(), (wj * yj).sum(), (wj * pj).sum(),
                          (wj * yj * yj).sum(), (wj * pj * pj).sum(), (wj * yj * pj).sum()]

    def result(self):
        out = {}
        for j, name in enumerate(TARGET_COLUMNS):
            W, wy, wp, wyy, wpp, wyp = self.s[j]
            if W < 1e-8:
                out[name] = 0.0
                continue
            my, mp = wy / W, wp / W
            cov = wyp / W - my * mp
            vy, vp = wyy / W - my * my, wpp / W - mp * mp
            out[name] = float(np.clip(cov / np.sqrt(vy * vp), -1.0, 1.0)) if vy > 1e-16 and vp > 1e-16 else 0.0
        out["weighted_pearson"] = (out["t0"] + out["t1"]) / 2
        return out


# ----------------------------------------------------------------------------- modelo
class RNNModel(nn.Module):
    """Red rápida (GRU/LSTM por fila) + opcionalmente una red LENTA que corre 1 de cada `slow_every` filas.

    La lenta recibe el promedio de los estados de la rápida durante la ventana y devuelve un vector de
    contexto que se concatena a la salida. Las filas de una ventana usan el contexto de la ventana ANTERIOR
    (causal, y calculable online en el kernel de inferencia).
    """

    def __init__(self, n_in: int, hidden: int, layers: int, cell: str, dropout: float = 0.0,
                 slow_hidden: int = 0, slow_every: int = 32, n_aux: int = 0):
        super().__init__()
        core = nn.GRU if cell == "gru" else nn.LSTM
        # dropout solo entre capas (PyTorch lo ignora con layers=1); no cambia la inferencia
        self.core = core(n_in, hidden, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.slow_hidden, self.slow_every = slow_hidden, slow_every
        self.n_aux = n_aux
        if slow_hidden:
            self.slow = nn.GRU(hidden, slow_hidden, num_layers=1, batch_first=True)
        self.head = nn.Linear(hidden + slow_hidden, 2 + n_aux)   # las salidas extra solo se usan al entrenar

    def forward(self, x, state=None, with_aux=False):
        core_state, slow_state, s_prev = state if state is not None else (None, None, None)
        out, core_state = self.core(x, core_state)
        if not self.slow_hidden:
            y = self.head(out)
            return (y if with_aux else y[..., :2]), (core_state, None, None)
        B, T, H = out.shape
        k = self.slow_every
        assert T % k == 0, f"el chunk ({T}) debe ser múltiplo de --slow-every ({k})"
        if s_prev is None:
            s_prev = out.new_zeros(B, self.slow_hidden)
        means = out.reshape(B, T // k, k, H).mean(2)                  # (B, W, H): resumen de cada ventana
        s_seq, slow_state = self.slow(means, slow_state)              # (B, W, S)
        s_use = torch.cat([s_prev.unsqueeze(1), s_seq[:, :-1]], 1)    # cada ventana usa el estado de la anterior
        s_broad = s_use.unsqueeze(2).expand(B, T // k, k, self.slow_hidden).reshape(B, T, self.slow_hidden)
        y = self.head(torch.cat([out, s_broad], -1))
        return (y if with_aux else y[..., :2]), (core_state, slow_state, s_seq[:, -1])


class TCNModel(nn.Module):
    """Convolucional causal dilatada. Mira una ventana larga de pasos con filtros aprendidos.

    Cada bloque es residual: h = h + gelu(norm(conv(h))). Las dilataciones crecen (1, 4, 16, 64, 256) para
    cubrir cientos de filas con pocas capas. El estado que se pasa entre trozos son las últimas
    (kernel-1)*dilatación entradas de cada bloque, que es lo mismo que el buffer circular de la inferencia.
    """

    def __init__(self, n_in: int, channels: int = 128, layers: int = 5, kernel: int = 3,
                 dropout: float = 0.0, n_aux: int = 0, dil_base: int = 4):
        super().__init__()
        self.kernel, self.n_aux = kernel, n_aux
        self.dils = [dil_base ** i for i in range(layers)]
        self.proj = nn.Linear(n_in, channels)
        self.convs = nn.ModuleList([nn.Conv1d(channels, channels, kernel, dilation=d) for d in self.dils])
        self.norms = nn.ModuleList([nn.LayerNorm(channels) for _ in self.dils])
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(channels, 2 + n_aux)
        self.receptive = 1 + (kernel - 1) * sum(self.dils)

    def forward(self, x, state=None, with_aux=False):
        h = self.proj(x)
        nuevo = []
        for i, (conv, d) in enumerate(zip(self.convs, self.dils)):
            pad = (self.kernel - 1) * d
            prev = state[i] if state is not None else h.new_zeros(h.shape[0], pad, h.shape[2])
            hin = torch.cat([prev, h], 1)
            nuevo.append(hin[:, hin.shape[1] - pad:])
            z = conv(hin.transpose(1, 2)).transpose(1, 2)
            h = h + self.drop(torch.nn.functional.gelu(self.norms[i](z)))
        y = self.head(h)
        return (y if with_aux else y[..., :2]), nuevo


def detach(state):
    if state is None:
        return None
    if isinstance(state, list):                                       # convolucional: un buffer por capa
        return [t.detach() for t in state]
    if isinstance(state, tuple) and len(state) == 3:                  # (estado rápido, estado lento, contexto)
        return tuple(detach(s) for s in state)
    return tuple(s.detach() for s in state) if isinstance(state, tuple) else state.detach()


AUX_NAMES = ["t0_suavizado20", "t1_suavizado20", "t0_suavizado100", "t1_suavizado100", "i1_movimiento10"]


def aux_targets(y, xraw, bidp1_0, askp1_0):
    """Targets auxiliares a partir de los targets y las features crudas de la secuencia COMPLETA.

    y: (B, T, 2) targets. xraw: (B, T, 112). Devuelve (B, T, 5), ya estandarizados.
    Mirar al futuro acá es legítimo: son targets, no entradas.
    """
    B, T, _ = y.shape
    outs = []
    for k in (20, 100):
        c = torch.cumsum(y.double(), 1)
        fut = (c[:, k:] - c[:, :-k]) / k                       # fut[t] = promedio de y[t+1 .. t+k]
        fut = torch.cat([fut, fut[:, -1:].expand(B, k, 2)], 1).float()
        outs.append(fut)
    mid1 = (xraw[:, :, askp1_0] + xraw[:, :, bidp1_0]) * 0.5    # precio medio del segundo instrumento
    mov = torch.cat([mid1[:, 10:] - mid1[:, :-10], torch.zeros(B, 10, device=y.device)], 1)
    outs.append(mov.unsqueeze(-1))
    a = torch.cat(outs, -1)
    return (a - a.mean((0, 1), keepdim=True)) / a.std((0, 1), keepdim=True).clamp_min(1e-6)


def loss_wmse(pred, target, mask, gamma=1.0):
    y = target.clamp(-2.0, 2.0)
    w = y.abs() * mask.unsqueeze(-1)
    if gamma != 1.0:
        w = w ** gamma
    return (w * (pred - y) ** 2).sum() / w.sum().clamp_min(1e-8)


def loss_gwp(pred, target, mask):
    """1 - WeightedPearson GLOBAL sobre todo el batch x tiempo (como la métrica), promedio de t0/t1."""
    y = target.clamp(-2.0, 2.0)
    w = (y.abs() * mask.unsqueeze(-1)).reshape(-1, 2)
    y = y.reshape(-1, 2)
    p = pred.reshape(-1, 2)
    ws = w.sum(0).clamp_min(1e-8)
    my, mp = (w * y).sum(0) / ws, (w * p).sum(0) / ws
    yc, pc = y - my, p - mp
    cov = (w * yc * pc).sum(0) / ws
    sy = ((w * yc * yc).sum(0) / ws).clamp_min(1e-8).sqrt()
    sp = ((w * pc * pc).sum(0) / ws).clamp_min(1e-8).sqrt()
    return (1.0 - cov / (sy * sp)).mean()


def loss_seqwp(pred, target, mask, stats, decay=0.98):
    """1 - WeightedPearson acumulado a lo largo de la secuencia (como la métrica), por target.

    stats: tensor (6, 2) con los estadísticos de los trozos anteriores (sin gradiente), o None al empezar
    una secuencia nueva. Devuelve (loss, stats_nuevos). decay<1 = promedio exponencial: el trozo actual
    pesa lo suficiente como para que el gradiente no se diluya al final de la secuencia.
    """
    y = target.clamp(-2.0, 2.0)
    w = (y.abs() * mask.unsqueeze(-1)).reshape(-1, 2)      # mismos pesos que la métrica: |y|
    y = y.reshape(-1, 2)
    p = pred.reshape(-1, 2)
    cur = torch.stack([w.sum(0), (w * y).sum(0), (w * p).sum(0),
                       (w * y * y).sum(0), (w * p * p).sum(0), (w * y * p).sum(0)])
    tot = cur if stats is None else cur + decay * stats
    W = tot[0].clamp_min(1e-8)
    my, mp = tot[1] / W, tot[2] / W
    cov = tot[5] / W - my * mp
    vy = (tot[3] / W - my * my).clamp_min(1e-10)
    vp = (tot[4] / W - mp * mp).clamp_min(1e-10)
    rho = cov / (vy * vp).sqrt()
    return (1.0 - rho).mean(), tot.detach()


def load_teacher(path, device):
    """Carga una GRU/LSTM de N capas (ambos formatos) como teacher congelado. Devuelve (net, mean, std, clip)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if "state_dict" in ck:
        sd, H, L, cell = ck["state_dict"], ck["config"]["hidden"], ck["config"]["layers"], ck["config"].get("cell", "gru")
        mean, std, clip = ck["pipeline"]["mean"], ck["pipeline"]["std"], float("inf")
    else:
        sd, H, L, cell = ck["model"], ck["architecture"]["hidden"], ck["architecture"]["layers"], "gru"
        mean, std, clip = ck["normalizer"]["mean"], ck["normalizer"]["std"], 20.0
    core = (nn.GRU if cell == "gru" else nn.LSTM)(112, H, num_layers=L, batch_first=True)
    net = nn.ModuleDict({"core": core, "head": nn.Linear(H, 2)}).to(device)
    net.load_state_dict(sd)
    net.eval()
    for prm in net.parameters():
        prm.requires_grad_(False)
    return net, torch.as_tensor(np.asarray(mean, np.float32), device=device), torch.as_tensor(np.asarray(std, np.float32), device=device), clip


def loss_wp(pred, target, mask):
    """-WeightedPearson por secuencia y target dentro del chunk (promedio)."""
    y = target.clamp(-2.0, 2.0)
    w = y.abs() * mask.unsqueeze(-1)                     # (B, T, 2)
    ws = w.sum(dim=1).clamp_min(1e-8)                     # (B, 2)
    my = (w * y).sum(1) / ws
    mp = (w * pred).sum(1) / ws
    yc, pc = y - my.unsqueeze(1), pred - mp.unsqueeze(1)
    cov = (w * yc * pc).sum(1) / ws
    sy = ((w * yc * yc).sum(1) / ws).clamp_min(1e-8).sqrt()
    sp = ((w * pc * pc).sum(1) / ws).clamp_min(1e-8).sqrt()
    rho = cov / (sy * sp)
    valid = (ws > 1e-6) & (sy > 1e-4)
    return -(rho * valid).sum() / valid.sum().clamp_min(1)


# ----------------------------------------------------------------------------- evaluación
@torch.no_grad()
def evaluate(model, pipe, groups: ParquetGroups, ids: list[int], device, batch: int):
    model.eval()
    acc, gacc = BlockAccumulator(), GlobalWP()
    rows = []
    for start in range(0, len(ids), batch):
        chunk_ids = ids[start:start + batch]
        data = [groups.read(i) for i in chunk_ids]
        x = torch.from_numpy(np.stack([pipe.transform(d[2]) for d in data])).to(device)
        pred = model(x)[0].float().cpu().numpy()
        for (seq_ix, need, _, y, scored), p in zip(data, pred):
            mask = need if scored is None else (need & scored)
            acc.add(y, p, mask)
            gacc.add(y, p, mask)
            yy, pp = y[mask], p[mask]
            if all(np.any(yy[:, j] > 0) and np.any(yy[:, j] < 0) for j in range(2)):
                w0, w1 = weighted_pearson(yy[:, 0], pp[:, 0]), weighted_pearson(yy[:, 1], pp[:, 1])
                rows.append({"seq_ix": seq_ix, "wp_t0": w0, "wp_t1": w1, "wp": (w0 + w1) / 2})
            else:
                rows.append({"seq_ix": seq_ix, "wp_t0": None, "wp_t1": None, "wp": None})
    model.train()
    res = acc.result()
    g = gacc.result()
    res.update(global_wp=g["weighted_pearson"], global_t0=g["t0"], global_t1=g["t1"], per_sequence_wp=res["weighted_pearson"])
    res["weighted_pearson"] = g["weighted_pearson"]   # la métrica GLOBAL es la oficial desde 2026-09
    res["t0"], res["t1"] = g["t0"], g["t1"]
    return res, rows


@torch.no_grad()
def _save_valid_preds(model, pipe, groups, ids, device, batch, path):
    model.eval()
    seqs, preds = [], []
    for start in range(0, len(ids), batch):
        data = [groups.read(i) for i in ids[start:start + batch]]
        x = torch.from_numpy(np.stack([pipe.transform(d[2]) for d in data])).to(device)
        p = model(x)[0].float().cpu().numpy().astype(np.float16)
        seqs += [d[0] for d in data]
        preds.append(p)
    model.train()
    np.savez_compressed(path, seq_ix=np.array(seqs, np.int64), preds=np.concatenate(preds))


@torch.no_grad()
def precompute_soft(teachers, teacher_paths, groups, ids, batch, workers, device, cache_path, chunk=2000):
    """Predicción promedio de los teachers para cada secuencia de ids -> dict id -> (SEQUENCE_LENGTH, 2) float16.
    Si cache_path existe y corresponde a los mismos teachers e ids, se carga; si no, se calcula y se guarda."""
    key = "|".join(Path(t).name for t in teacher_paths)
    if cache_path:
        cp = Path(cache_path)
        if cp.exists():
            z = np.load(cp, allow_pickle=False)
            if str(z["key"]) == key and z["ids"].tolist() == list(ids):
                soft, zids = z["soft"], z["ids"]   # leer UNA vez: NpzFile relee el archivo entero en cada acceso z["soft"]
                print(f"Soft targets cargados de {cp} ({soft.nbytes/1e6:.0f} MB)", flush=True)
                return {int(i): soft[k] for k, i in enumerate(zids)}
            print(f"Cache {cp} no coincide (teachers o ids distintos): se recalcula", flush=True)
    t0 = time.time()
    soft = np.zeros((len(ids), SEQUENCE_LENGTH, 2), np.float16)
    pool = ThreadPoolExecutor(max_workers=workers)
    chunks = [ids[s:s + batch] for s in range(0, len(ids), batch)]
    futures = {bi: pool.submit(lambda b: [groups.read(i) for i in b], chunks[bi]) for bi in range(min(workers, len(chunks)))}
    pos = 0
    for bi in range(len(chunks)):
        data = futures.pop(bi).result()
        if bi + workers < len(chunks):
            futures[bi + workers] = pool.submit(lambda b: [groups.read(i) for i in b], chunks[bi + workers])
        xraw = torch.from_numpy(np.stack([d[2] for d in data])).to(device)
        acc = torch.zeros(len(data), SEQUENCE_LENGTH, 2, device=device)
        for tnet, tmean, tstd, tclip in teachers:
            zt = ((xraw - tmean) / tstd).clamp(-tclip, tclip)
            state = None
            for s0 in range(0, SEQUENCE_LENGTH, chunk):   # por trozos con estado: 20.000 pasos de una vez pide >7 GB en GPU
                out_t, state = tnet["core"](zt[:, s0:s0 + chunk], state)
                acc[:, s0:s0 + chunk] += tnet["head"](out_t)
        soft[pos:pos + len(data)] = (acc / len(teachers)).half().cpu().numpy()
        pos += len(data)
        if bi % max(1, len(chunks) // 10) == 0 or bi == len(chunks) - 1:
            print(f"  soft targets {pos}/{len(ids)} ({(time.time()-t0)/60:.1f} min)", flush=True)
    pool.shutdown()
    if cache_path:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        np.savez(cache_path, soft=soft, ids=np.asarray(ids, np.int64), key=np.array(key))
        print(f"Soft targets guardados en {cache_path} ({soft.nbytes/1e6:.0f} MB, {(time.time()-t0)/60:.1f} min)", flush=True)
    return {int(i): soft[k] for k, i in enumerate(ids)}


# ----------------------------------------------------------------------------- entrenamiento
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--train-parquet", nargs="+", default=None, help="default: datasets/train.parquet local; con --resume, la ruta del checkpoint")
    ap.add_argument("--valid-parquet", nargs="+", default=None, help="default: datasets/valid.parquet local; con --resume, la ruta del checkpoint")
    ap.add_argument("--train-groups", default="500", help="N o 'all'")
    ap.add_argument("--valid-groups", default="10", help="N o 'all' (las 1.873)")
    ap.add_argument("--seed", type=int, default=2026, help="selección de grupos (valid usa seed+1, como FirstTry)")
    ap.add_argument("--model-seed", type=int, default=2026)
    ap.add_argument("--cell", choices=["gru", "lstm"], default="gru")
    ap.add_argument("--hidden", type=int, default=128)
    ap.add_argument("--layers", type=int, default=2)
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--batch", type=int, default=None, help="secuencias por batch (default: 32 GPU / 8 CPU)")
    ap.add_argument("--chunk", type=int, default=None, help="pasos por chunk de BPTT (default: 512 GPU / 256 CPU)")
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--lr-decay", type=float, default=1.0, help="factor multiplicativo por época")
    ap.add_argument("--clip", type=float, default=1.0, help="gradient clipping (0 = off)")
    ap.add_argument("--dropout", type=float, default=0.0, help="dropout entre capas de la GRU (regularización)")
    ap.add_argument("--mask-coef", default=None, help="JSON con el modelo que estima la probabilidad de que una fila "
                    "sea puntuada por la métrica (mean/std/w/b). Pesa el entrenamiento hacia esas filas")
    ap.add_argument("--mask-power", type=float, default=1.0, help="con --mask-coef: exponente del peso (0 = sin efecto)")
    ap.add_argument("--arch", choices=["rnn", "tcn"], default="rnn", help="rnn: GRU/LSTM (lo de siempre); "
                    "tcn: convolucional causal dilatada, otra clase de modelo")
    ap.add_argument("--tcn-channels", type=int, default=128)
    ap.add_argument("--tcn-layers", type=int, default=5, help="dilataciones 1, 4, 16, 64, 256: campo receptivo 683 filas")
    ap.add_argument("--tcn-kernel", type=int, default=3)
    ap.add_argument("--train-half", type=int, choices=[1, 2], default=0, help="entrenar solo con la mitad 1 o la "
                    "mitad 2 de las secuencias de train (disjuntas): fuerza diversidad entre modelos")
    ap.add_argument("--aux-targets", action="store_true", help="cabezas extra al entrenar: targets suavizados a 20 y "
                    "100 filas + movimiento del segundo instrumento. Dan una señal mucho menos ruidosa que el target "
                    "crudo; se recortan al guardar, así que no cuestan nada en inferencia")
    ap.add_argument("--aux-weight", type=float, default=0.3, help="peso de las tareas auxiliares en la loss")
    ap.add_argument("--slow-hidden", type=int, default=0, help="capa recurrente LENTA de N unidades que corre 1 de cada "
                    "--slow-every filas (0 = off): memoria larga barata y errores distintos a los de la familia actual")
    ap.add_argument("--slow-every", type=int, default=32, help="cada cuántas filas corre la capa lenta (divisor del chunk)")
    ap.add_argument("--weight-decay", type=float, default=0.0, help="AdamW weight decay (regularización)")
    ap.add_argument("--swa-start", type=int, default=0, help="promediar pesos desde esta época (0 = off); evalúa y guarda swa_model.pt")
    ap.add_argument("--save-epochs", action="store_true", help="guardar epoch_N.pt de cada época (para promediar offline)")
    ap.add_argument("--loss", choices=["wmse", "wp", "gwp", "seqwp"], default="wmse",
                    help="wmse: error cuadrático alineado; wp: 1-WP por secuencia (chunk); gwp: 1-WP GLOBAL del batch; "
                         "seqwp: 1-WP acumulado a lo largo de toda la secuencia (lo más parecido a la métrica)")
    ap.add_argument("--seqwp-decay", type=float, default=0.98, help="con --loss seqwp: olvido de los estadísticos por trozo")
    ap.add_argument("--wmse-anchor", type=float, default=0.0, help="con --loss seqwp: suma este peso por WMSE, para que la "
                    "escala de las predicciones no quede libre (la correlación es invariante a escala)")
    ap.add_argument("--gwp-weight", type=float, default=0.0, help="suma gwp_weight * (1 - WP global del batch) a la loss")
    ap.add_argument("--weight-power", type=float, default=1.0, help="pesos |y|^gamma en la WMSE (1 = métrica)")
    ap.add_argument("--lr-schedule", choices=["step", "cosine"], default="step", help="step: lr*decay^epoch; cosine: de lr a lr-min")
    ap.add_argument("--lr-min", type=float, default=1e-5)
    ap.add_argument("--init-from", default=None, help="checkpoint (best_model.pt / swa_model.pt) para inicializar pesos (fine-tuning)")
    ap.add_argument("--distill-teachers", nargs="*", default=[], help="rutas .pt de teachers; sus predicciones promediadas son target blando")
    ap.add_argument("--distill-weight", type=float, default=1.0, help="peso de la loss blanda (WMSE contra el teacher)")
    ap.add_argument("--distill-cache", default=None, help="archivo .npz para precalcular UNA vez las predicciones de los teachers "
                    "(fp16, ~850 MB para todo train) y reutilizarlas en todas las épocas/corridas; '' = precalcular solo en RAM. "
                    "Sin esta opción los teachers corren en cada chunk de cada época (2x más lento)")
    ap.add_argument("--save-valid-preds", action="store_true", help="guardar predicciones del mejor modelo en valid (npz float16)")
    ap.add_argument("--top-k", type=int, default=0, help="lags/deltas por target de features.py (0,3,5,10,20)")
    ap.add_argument("--ranking", default=str((HERE.parents[1] if len(HERE.parents) > 1 else HERE) / "explore" / "feature_select" / "results" / "top_20_by_target.csv"))
    ap.add_argument("--sentinel-flags", action="store_true", help="a5/a6/a7: -5.1993 -> 0 + columna indicadora")
    ap.add_argument("--ema-features", action="store_true", help="memoria larga: desvío respecto de promedios móviles de "
                    "100/1.000/10.000 filas (12 columnas) + volatilidad lenta. El BPTT truncado en 512 pasos impide que "
                    "la red aprenda esto sola; en inferencia cuesta una cuenta por fila")
    ap.add_argument("--cross-features", action="store_true", help="agrega 24 combinaciones entre columnas de la misma fila "
                    "(spread, mid, huecos entre niveles, desequilibrio de volumen, i0 vs i1); causales y baratas en inferencia")
    ap.add_argument("--norm-rows", type=int, default=1000, help="filas por grupo (máx 200 grupos) para el normalizador")
    ap.add_argument("--eval-batch", type=int, default=16)
    ap.add_argument("--eval-every", type=int, default=0, help="además evaluar cada N batches (0 = solo por época)")
    ap.add_argument("--workers", type=int, default=2, help="hilos de lectura de parquet")
    ap.add_argument("--threads", type=int, default=0, help="hilos de torch en CPU (0 = auto)")
    ap.add_argument("--output-dir", default=str(HERE / "results"))
    ap.add_argument("--run-name", default=None)
    ap.add_argument("--resume", default=None, help="carpeta de un run para retomar (usa checkpoint.pt)")
    ap.add_argument("--resume-epochs", type=int, default=None, help="con --resume: seguir hasta esta época (pisa el epochs del checkpoint)")
    ap.add_argument("--resume-swa-start", type=int, default=None, help="con --resume: SWA desde esta época (el promedio anterior no se conserva)")
    ap.add_argument("--resume-cache", default=None, help="con --resume: ruta del cache de soft targets en esta sesión (pisa distill_cache del checkpoint)")
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    on_gpu = device.type == "cuda"
    batch = args.batch or (32 if on_gpu else 8)
    chunk = args.chunk or (512 if on_gpu else 256)
    if not on_gpu:
        torch.set_num_threads(args.threads or max(1, (os.cpu_count() or 2) - 2))
    print(f"device={device}  batch={batch}  chunk={chunk}  torch={torch.__version__}", flush=True)

    if args.resume:   # las rutas de datos del run original, salvo que se pasen explícitas (en Kaggle no existe la ruta local)
        ck0 = torch.load(Path(args.resume) / "checkpoint.pt", map_location="cpu", weights_only=False)["config"]
        args.train_parquet = args.train_parquet or ck0["train_parquet"]
        args.valid_parquet = args.valid_parquet or ck0["valid_parquet"]
        del ck0
    args.train_parquet = args.train_parquet or [str(ROOT / "datasets" / "train.parquet")]
    args.valid_parquet = args.valid_parquet or [str(ROOT / "datasets" / "valid.parquet")]
    train = ParquetGroups(args.train_parquet)
    valid = ParquetGroups(args.valid_parquet)
    if not valid.has_scored:
        raise ValueError("valid necesita is_scored para la métrica oficial")
    train_ids = list(range(len(train))) if args.train_groups == "all" else select_groups(len(train), int(args.train_groups), args.seed)
    if args.train_half:     # mitades disjuntas, repartidas al azar pero con semilla fija para que sean reproducibles
        orden = np.random.default_rng(777).permutation(len(train_ids))
        corte = len(train_ids) // 2
        elegidas = orden[:corte] if args.train_half == 1 else orden[corte:]
        train_ids = [train_ids[i] for i in sorted(elegidas)]
        print(f"Mitad {args.train_half}: {len(train_ids)} secuencias de entrenamiento", flush=True)
    valid_ids = list(range(len(valid))) if str(args.valid_groups) == "all" else select_groups(len(valid), int(args.valid_groups), args.seed + 1)

    # ---- run dir / resume
    if args.resume:
        run_dir = Path(args.resume)
        ck = torch.load(run_dir / "checkpoint.pt", map_location="cpu", weights_only=False)
        cfg = ck["config"]
        pipe = FeaturePipeline(cfg["top_k"], cfg["sentinel_flags"], Path(cfg["ranking"]), cfg.get("cross_features", False),
                               cfg.get("ema_features", False))
        pipe.mean, pipe.std = ck["pipeline"]["mean"], ck["pipeline"]["std"]
        if args.resume_epochs is not None:
            cfg["epochs"] = args.resume_epochs
        if args.resume_swa_start is not None:
            cfg["swa_start"] = args.resume_swa_start
        if args.resume_cache is not None:
            cfg["distill_cache"] = args.resume_cache
        train_ids, valid_ids = list(cfg["train_ids"]), list(cfg["valid_ids"])   # mismos datos que el run original
        batch, chunk = cfg["batch"], cfg["chunk"]                                 # mismo batch/chunk que el run original
        print(f"Retomando {run_dir} desde época {ck['epoch']} batch {ck['batch']} hasta la época {cfg['epochs']} (SWA desde {cfg.get('swa_start', 0)})", flush=True)
    else:
        name = args.run_name or datetime.now().strftime("run_%Y%m%d_%H%M%S")
        run_dir = Path(args.output_dir) / name
        run_dir.mkdir(parents=True, exist_ok=False)
        cfg = {**vars(args), "device": str(device), "batch": batch, "chunk": chunk,
               "train_ids": train_ids, "valid_ids": valid_ids, "n_train": len(train_ids)}
        pipe = FeaturePipeline(args.top_k, args.sentinel_flags, Path(args.ranking), args.cross_features, args.ema_features)
        t0 = time.time()
        if len(train_ids) > 200:
            norm_ids = [train_ids[i] for i in select_groups(len(train_ids), 200, args.seed + 7)]
        else:
            norm_ids = train_ids
        pipe.fit_normalizer(train, norm_ids, args.norm_rows, args.seed + 2)
        print(f"Normalizador: {len(norm_ids)} grupos, {len(pipe.names)} entradas ({time.time()-t0:.0f}s)", flush=True)
        ck = None
        (run_dir / "run_config.json").write_text(json.dumps(cfg, indent=2, default=str), encoding="utf-8")

    torch.manual_seed(cfg["model_seed"])
    n_aux = len(AUX_NAMES) if cfg.get("aux_targets") else 0
    if cfg.get("arch", "rnn") == "tcn":
        model = TCNModel(len(pipe.names), cfg["tcn_channels"], cfg["tcn_layers"], cfg["tcn_kernel"],
                         cfg.get("dropout", 0.0), n_aux).to(device)
    else:
        model = RNNModel(len(pipe.names), cfg["hidden"], cfg["layers"], cfg["cell"], cfg.get("dropout", 0.0),
                         cfg.get("slow_hidden", 0), cfg.get("slow_every", 32), n_aux).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg.get("weight_decay", 0.0))
    loss_fn = {"wmse": loss_wmse, "wp": loss_wp, "gwp": loss_gwp, "seqwp": None}[cfg["loss"]]
    if cfg.get("init_from") and not ck:
        init = torch.load(cfg["init_from"], map_location="cpu", weights_only=False)
        model.load_state_dict(init["state_dict"] if "state_dict" in init else init["model"])
        if "pipeline" in init:   # usar el normalizador del modelo inicial (los pesos lo asumen)
            pipe.mean, pipe.std = init["pipeline"]["mean"], init["pipeline"]["std"]
        print(f"Pesos inicializados desde {cfg['init_from']}", flush=True)
    teachers = [load_teacher(t, device) for t in cfg.get("distill_teachers", []) or []]
    soft_cache = None
    if teachers:
        print(f"Destilación: {len(teachers)} teachers, peso {cfg['distill_weight']}", flush=True)
        if cfg.get("distill_cache") is not None:
            soft_cache = precompute_soft(teachers, cfg["distill_teachers"], train, train_ids, batch, cfg["workers"], device, cfg["distill_cache"] or None)
            del teachers
            teachers = []
            if device.type == "cuda":
                torch.cuda.empty_cache()
    mask_mu = mask_sd = mask_w = mask_b = None
    if cfg.get("mask_coef"):
        mc = json.loads(Path(cfg["mask_coef"]).read_text(encoding="utf-8"))
        mask_mu = torch.as_tensor(np.asarray(mc["mean"], np.float32), device=device)
        mask_sd = torch.as_tensor(np.asarray(mc["std"], np.float32), device=device)
        mask_w = torch.as_tensor(np.asarray(mc["w"], np.float32), device=device)
        mask_b = float(mc["b"])
        print(f"Pesado por probabilidad de ser puntuada (AUC {mc.get('auc', 0):.3f}), potencia {cfg['mask_power']}", flush=True)

    n_params = sum(p.numel() for p in model.parameters())
    if cfg.get("arch", "rnn") == "tcn":
        macs = len(pipe.names) * cfg["tcn_channels"] + cfg["tcn_layers"] * cfg["tcn_kernel"] * cfg["tcn_channels"] ** 2
        print(f"TCN {cfg['tcn_channels']} canales x{cfg['tcn_layers']} capas  campo receptivo={model.receptive} filas  "
              f"entradas={len(pipe.names)}  params={n_params:,}  ~{macs:,} operaciones por fila "
              f"(la LSTM 128x2 usa 254.208)", flush=True)
    else:
        print(f"{cfg['cell'].upper()} h={cfg['hidden']} x{cfg['layers']}  entradas={len(pipe.names)}  params={n_params:,}", flush=True)

    start_epoch, start_batch, history, best = 0, 0, [], -1.0
    if ck:
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["optimizer"])
        start_epoch, start_batch, history, best = ck["epoch"], ck["batch"], ck["history"], ck["best"]
        torch.set_rng_state(ck["rng"]["torch"])

    def save_checkpoint(epoch, b_idx):
        torch.save({"model": model.state_dict(), "optimizer": opt.state_dict(), "epoch": epoch, "batch": b_idx,
                    "history": history, "best": best, "config": cfg, "pipeline": pipe.state(),
                    "rng": {"torch": torch.get_rng_state()}}, run_dir / "checkpoint.pt")

    def save_model(path, state_dict=None, extra=None):
        sd = dict(state_dict if state_dict is not None else model.state_dict())
        if cfg.get("aux_targets"):      # el modelo guardado tiene solo las 2 salidas reales
            sd["head.weight"], sd["head.bias"] = sd["head.weight"][:2], sd["head.bias"][:2]
        torch.save({"state_dict": {k: v.detach().cpu() for k, v in sd.items()},
                    "config": {k: cfg[k] for k in ("cell", "hidden", "layers")} |
                              {k: cfg.get(k) for k in ("arch", "tcn_channels", "tcn_layers", "tcn_kernel")} |
                              {k: cfg.get(k, 0 if k == "slow_hidden" else 32) for k in ("slow_hidden", "slow_every")},
                    "pipeline": pipe.state(), "n_inputs": len(pipe.names), **(extra or {})}, path)

    swa_sum, swa_n = None, 0

    def run_eval(tag, epoch, b_idx):
        nonlocal best
        t = time.time()
        res, rows = evaluate(model, pipe, valid, valid_ids, device, cfg["eval_batch"])
        wp = res["weighted_pearson"]
        print(f"  [{tag}] valid GLOBAL WP={wp:.4f} (t0={res['t0']:.4f}, t1={res['t1']:.4f})   "
              f"[por secuencia: {res['per_sequence_wp']:.4f}]  ({time.time()-t:.0f}s)", flush=True)
        history.append({"epoch": epoch, "batch": b_idx, "tag": tag, "wp": wp, "t0": res["t0"], "t1": res["t1"], "time": time.time()})
        if wp > best:
            best = wp
            save_model(run_dir / "best_model.pt")
            if cfg.get("save_valid_preds"):
                _save_valid_preds(model, pipe, valid, valid_ids, device, cfg["eval_batch"], run_dir / "valid_preds_best.npz")
            with (run_dir / "best_by_rowgroup.csv").open("w", newline="", encoding="utf-8") as f:
                w = csv.DictWriter(f, fieldnames=list(rows[0]))
                w.writeheader()
                w.writerows(rows)
        (run_dir / "metrics.json").write_text(json.dumps({"best_valid_wp": best, "last": res, "history": history,
                                                          "config": cfg, "params": n_params}, indent=2, default=str), encoding="utf-8")
        return wp

    # ---- loop
    pool = ThreadPoolExecutor(max_workers=cfg["workers"])
    n_batches = math.ceil(len(train_ids) / batch)
    warm = torch.zeros(SEQUENCE_LENGTH, dtype=torch.float32)
    warm[WARMUP:] = 1.0
    t_start = time.time()
    done_batches = 0
    for epoch in range(start_epoch, cfg["epochs"]):
        order = np.random.default_rng(cfg["seed"] + 100 + epoch).permutation(len(train_ids))
        batches = [[train_ids[i] for i in order[s:s + batch]] for s in range(0, len(order), batch)]
        if cfg.get("lr_schedule", "step") == "cosine":
            lr_now = cfg["lr_min"] + 0.5 * (cfg["lr"] - cfg["lr_min"]) * (1 + math.cos(math.pi * epoch / max(1, cfg["epochs"] - 1)))
        else:
            lr_now = cfg["lr"] * (cfg["lr_decay"] ** epoch)
        for g in opt.param_groups:
            g["lr"] = lr_now
        print(f"época {epoch+1}: lr={lr_now:.2e}", flush=True)

        def load(bi):
            return [train.read(i) for i in batches[bi]]

        first = start_batch if epoch == start_epoch else 0
        futures = {bi: pool.submit(load, bi) for bi in range(first, min(first + cfg["workers"], n_batches))}
        loss_acc, loss_n = 0.0, 0
        for bi in range(first, n_batches):
            data = futures.pop(bi).result()
            nxt = bi + cfg["workers"]
            if nxt < n_batches:
                futures[nxt] = pool.submit(load, nxt)
            x = torch.from_numpy(np.stack([pipe.transform(d[2]) for d in data])).to(device, non_blocking=True)
            y = torch.from_numpy(np.stack([d[3] for d in data])).to(device, non_blocking=True)
            m = warm.unsqueeze(0).expand(len(data), -1).to(device)
            if mask_w is not None:          # pesar hacia las filas que la métrica va a mirar
                xr0 = torch.from_numpy(np.stack([d[2] for d in data])).to(device, non_blocking=True)
                pr = torch.sigmoid(((xr0 - mask_mu) / mask_sd) @ mask_w + mask_b)
                m = m * (pr / pr.mean()).pow(cfg["mask_power"])
                del xr0
            state = None
            seq_stats = None                      # estadísticos acumulados de la secuencia (--loss seqwp)
            aux_full = None
            if cfg.get("aux_targets"):            # se calculan sobre la secuencia entera, antes de cortarla en trozos
                xr = torch.from_numpy(np.stack([d[2] for d in data])).to(device, non_blocking=True)
                aux_full = aux_targets(y, xr, BIDP1[0], ASKP1[0])
                del xr
            if teachers:
                xraw = torch.from_numpy(np.stack([d[2] for d in data])).to(device, non_blocking=True)
                t_states = [None] * len(teachers)
            if soft_cache is not None:
                soft_full = torch.from_numpy(np.stack([soft_cache[i] for i in batches[bi]])).to(device, non_blocking=True).float()
            for s in range(0, SEQUENCE_LENGTH, chunk):
                e = min(s + chunk, SEQUENCE_LENGTH)
                pred, state = model(x[:, s:e], state, with_aux=aux_full is not None)
                if aux_full is not None:
                    pred, pred_aux = pred[..., :2], pred[..., 2:]
                if soft_cache is not None:
                    soft = soft_full[:, s:e]
                elif teachers:
                    with torch.no_grad():
                        soft = 0.0
                        for ti, (tnet, tmean, tstd, tclip) in enumerate(teachers):
                            zt = ((xraw[:, s:e] - tmean) / tstd).clamp(-tclip, tclip)
                            out_t, t_states[ti] = tnet["core"](zt, t_states[ti])
                            soft = soft + tnet["head"](out_t)
                        soft = soft / len(teachers)
                if e > WARMUP:
                    if cfg["loss"] == "wmse":
                        loss = loss_wmse(pred, y[:, s:e], m[:, s:e], cfg.get("weight_power", 1.0))
                    elif cfg["loss"] == "seqwp":
                        loss, seq_stats = loss_seqwp(pred, y[:, s:e], m[:, s:e], seq_stats, cfg.get("seqwp_decay", 0.98))
                        if cfg.get("wmse_anchor", 0.0) > 0:
                            loss = loss + cfg["wmse_anchor"] * loss_wmse(pred, y[:, s:e], m[:, s:e])
                    else:
                        loss = loss_fn(pred, y[:, s:e], m[:, s:e])
                    if cfg.get("gwp_weight", 0.0) > 0:
                        loss = loss + cfg["gwp_weight"] * loss_gwp(pred, y[:, s:e], m[:, s:e])
                    if teachers or soft_cache is not None:
                        wsoft = y[:, s:e].clamp(-2.0, 2.0).abs() * m[:, s:e].unsqueeze(-1)   # mismos pesos que la métrica
                        loss = loss + cfg["distill_weight"] * ((wsoft * (pred - soft) ** 2).sum() / wsoft.sum().clamp_min(1e-8))
                    if aux_full is not None:
                        ma = m[:, s:e].unsqueeze(-1)
                        loss = loss + cfg["aux_weight"] * ((ma * (pred_aux - aux_full[:, s:e]) ** 2).sum()
                                                           / ma.sum().clamp_min(1e-8) / len(AUX_NAMES))
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    if cfg["clip"] > 0:
                        nn.utils.clip_grad_norm_(model.parameters(), cfg["clip"])
                    opt.step()
                    loss_acc += float(loss.detach())
                    loss_n += 1
                state = detach(state)
            done_batches += 1
            if bi % max(1, n_batches // 20) == 0 or bi == n_batches - 1:
                el = time.time() - t_start
                total = (cfg["epochs"] - start_epoch) * n_batches - (start_batch if epoch == start_epoch else 0)
                eta = el / max(1, done_batches) * (total - done_batches)
                print(f"época {epoch+1}/{cfg['epochs']} batch {bi+1}/{n_batches}  loss={loss_acc/max(1,loss_n):.4f}  "
                      f"transcurrido {el/60:.1f} min  restante ~{eta/60:.1f} min", flush=True)
                loss_acc, loss_n = 0.0, 0
            if cfg["eval_every"] and (bi + 1) % cfg["eval_every"] == 0 and bi + 1 < n_batches:
                run_eval(f"e{epoch+1}b{bi+1}", epoch, bi + 1)
                save_checkpoint(epoch, bi + 1)
        run_eval(f"época {epoch+1}", epoch + 1, 0)
        save_checkpoint(epoch + 1, 0)
        if cfg.get("save_epochs"):
            save_model(run_dir / f"epoch_{epoch+1:02d}.pt", extra={"epoch": epoch + 1})
        if cfg.get("swa_start", 0) and epoch + 1 >= cfg["swa_start"]:
            cur = {k: v.detach().float().cpu().clone() for k, v in model.state_dict().items()}
            if swa_sum is None:
                swa_sum = cur
            else:
                for k in swa_sum:
                    swa_sum[k] += cur[k]
            swa_n += 1
        start_batch = 0
    pool.shutdown()
    save_model(run_dir / "last_model.pt")
    if swa_sum is not None and swa_n > 1:
        swa_sd = {k: (v / swa_n).to(device) for k, v in swa_sum.items()}
        backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(swa_sd)
        res, _ = evaluate(model, pipe, valid, valid_ids, device, cfg["eval_batch"])
        print(f"  [SWA épocas {cfg['swa_start']}-{cfg['epochs']} ({swa_n} modelos)] valid GLOBAL WP={res['weighted_pearson']:.4f} "
              f"(t0={res['t0']:.4f}, t1={res['t1']:.4f})", flush=True)
        save_model(run_dir / "swa_model.pt", extra={"swa": {"start": cfg["swa_start"], "n": swa_n, "valid": res}})
        if cfg.get("save_valid_preds"):   # permite armar tablas de ensambles en la laptop, sin GPU
            _save_valid_preds(model, pipe, valid, valid_ids, device, cfg["eval_batch"], run_dir / "valid_preds_swa.npz")
        history.append({"epoch": cfg["epochs"], "batch": 0, "tag": "swa", "wp": res["weighted_pearson"], "t0": res["t0"], "t1": res["t1"], "time": time.time()})
        (run_dir / "metrics.json").write_text(json.dumps({"best_valid_wp": best, "swa_valid_wp": res["weighted_pearson"], "history": history,
                                                          "config": cfg, "params": n_params}, indent=2, default=str), encoding="utf-8")
        model.load_state_dict(backup)
    print(f"Mejor valid WP: {best:.4f}   Resultados en {run_dir.resolve()}", flush=True)


if __name__ == "__main__":
    main()


## Coeficientes del modelo de la máscara

In [ ]:
import json
coef = {"mean": [-1.547656536102295, 0.16519589722156525, 0.4057227671146393, -1.7399582862854004, -0.1136803850531578, 0.8371194005012512, -0.3338816463947296, -0.10603488981723785, -0.8204420208930969, -0.4485017955303192, 1.0521769523620605, -0.6776139140129089, 0.2934568226337433, 0.5483248829841614, -1.155493974685669, -0.5466122031211853, 1.4597574472427368, 0.06582676619291306, 0.6649800539016724, 1.9605714082717896, 1.1491931676864624, -1.0148520469665527, 1.352736234664917, 0.0995887741446495, -0.8563451766967773, -1.2388697862625122, 1.153847098350525, 0.6819772124290466, 0.6814267039299011, 0.13290487229824066, 0.9180064797401428, -0.9216588139533997, 0.5573667883872986, -0.4764370918273926, -0.1638641506433487, 0.08018238097429276, -0.6754230856895447, -0.9345325827598572, -0.49624794721603394, -1.3424782752990723, -0.6743473410606384, 0.5202140808105469, 0.5753893256187439, -0.4179791808128357, -1.0125657320022583, 0.16613231599330902, -0.3040018081665039, 1.0043983459472656, 0.020068811252713203, 0.10446194559335709, -0.07023195922374725, 0.026683542877435684, -1.591835618019104, 0.16396991908550262, 0.4053734540939331, -1.7726157903671265, -0.11667587608098984, 0.8462523221969604, -0.34265685081481934, -0.11639177799224854, -0.8362901210784912, -0.45284172892570496, 1.0701289176940918, -0.690758228302002, 0.28702113032341003, 0.5491006374359131, -1.1667789220809937, -0.5660532116889954, 1.4716899394989014, 0.061492919921875, 0.6695289611816406, 1.9861117601394653, 1.1649506092071533, -1.0376017093658447, 1.1883970499038696, 0.2992528975009918, -0.6153140664100647, -1.015643835067749, 1.0976817607879639, 0.5854092240333557, 0.6820551156997681, 0.1420326828956604, 0.861485481262207, -0.5747561454772949, 0.46255552768707275, -0.2168104648590088, -0.13256198167800903, -0.055083293467760086, -0.48200902342796326, -0.7513952255249023, -0.4175654947757721, -1.0981508493423462, -0.6134722828865051, 0.42830950021743774, 0.45324406027793884, -0.18355809152126312, -0.8759312033653259, 0.19174319505691528, -0.23259639739990234, 0.9645450115203857, -0.02686101384460926, 0.15265677869319916, -0.2029026597738266, -0.006475934758782387, -0.05637090653181076, -0.001387768192216754, -0.21936611831188202, -0.31897708773612976, 0.08589234203100204, -0.19128531217575073, -0.7570308446884155, -1.9021058082580566], "std": [0.3181327283382416, 0.05317571386694908, 0.06080365180969238, 0.28124743700027466, 0.08339795470237732, 0.08274330198764801, 0.14244920015335083, 0.07275639474391937, 0.15664450824260712, 0.24509300291538239, 0.17009234428405762, 0.2149135172367096, 0.03699441999197006, 0.07620476931333542, 0.22302210330963135, 0.12992121279239655, 0.15969692170619965, 0.04143407568335533, 0.05898550897836685, 0.19577859342098236, 0.09637738019227982, 0.214300736784935, 0.7000667452812195, 0.7500686645507812, 0.7574436068534851, 0.6384292840957642, 0.7411064505577087, 0.8458328247070312, 0.7992108464241028, 0.8347058296203613, 0.7365319728851318, 0.6624628901481628, 0.6754627227783203, 0.7379180788993835, 0.6936625242233276, 0.8643292784690857, 0.662463903427124, 0.7380217909812927, 0.7301239371299744, 0.6252932548522949, 0.6740785837173462, 0.816116213798523, 0.7842554450035095, 0.6799810528755188, 0.539658784866333, 0.5492729544639587, 0.6742620468139648, 0.6728093028068542, 0.5975139737129211, 0.5590482354164124, 0.56223464012146, 0.6061731576919556, 0.36287611722946167, 0.05764702707529068, 0.06315144151449203, 0.3228132128715515, 0.08602175861597061, 0.09594304859638214, 0.15109996497631073, 0.0829954519867897, 0.17912593483924866, 0.24762021005153656, 0.1850614845752716, 0.23255860805511475, 0.046039436012506485, 0.07615002244710922, 0.24677148461341858, 0.1555638611316681, 0.1744384914636612, 0.04784815385937691, 0.06394822150468826, 0.21448631584644318, 0.11239191889762878, 0.2407115399837494, 0.6104733347892761, 0.7189717888832092, 0.7308701276779175, 0.645473062992096, 0.760942816734314, 0.785469651222229, 0.8612520098686218, 0.8602679967880249, 0.7801787853240967, 0.641737163066864, 0.6360907554626465, 0.7912012934684753, 0.7688268423080444, 0.8995649218559265, 0.644740104675293, 0.814046323299408, 0.8543609976768494, 0.5318410992622375, 0.8278532028198242, 0.8118370771408081, 0.782884418964386, 0.7890680432319641, 0.7282891869544983, 0.5655069947242737, 0.7791793942451477, 0.6972135305404663, 0.8303688764572144, 0.7911268472671509, 0.763051450252533, 0.8409874439239502, 0.9263263940811157, 0.969357430934906, 0.8392751812934875, 0.8659502267837524, 0.9253749251365662, 1.9010443687438965, 2.128687620162964, 2.9279632568359375], "w": [0.04902592004487209, 0.1977042026417566, 0.19431326584446212, 0.22782569570834252, -0.18525134662945678, 0.015990648879639944, 0.17372576621422955, -0.03391013509257002, 0.07465461048518775, -0.5130948213977738, 0.28777555966220597, -0.23731261689782943, 0.07153025084605498, 0.012635444307423064, 0.022984354312011842, 0.2629300068577309, 0.2548546759532778, -0.10423607112984476, -0.01293367196259335, -0.1991582954942988, 0.09357431069287937, -0.018304969430891455, 0.1075721910675032, -0.021494714572106517, -0.08159709174006594, -0.007859682451713963, 0.048291088966041826, 0.051645287677994395, 0.03741005877431032, 0.03308638139627121, 0.047416496672061825, -0.19052481744309047, -0.021727134838812934, -0.057708896859843276, -0.0022366981055038466, 0.04355898696696161, -0.07632241126061536, -0.009878167568866694, 0.002098452662239873, -0.03792153919783839, 0.08542795161679206, 0.03552513795210549, 0.05089535410557413, 0.07130928279902932, -0.007283958679572049, 0.05713089000342782, 0.017146144145623904, 0.024592965688352553, -0.10888103421460942, 0.0810607579979276, -0.10178815301767609, -0.03202785584029869, -0.02222571608150439, 0.05079173537347353, 0.1272270282626755, 0.3225949426398112, 0.11006962013592625, 0.13300732591040645, -0.11375696085476071, -0.04989829578687998, 0.13117223121583985, -0.2761474805548066, 0.0573225101876018, -0.09695573713138594, 0.07646392303841239, 0.09821530247214512, 0.17819683763916694, -0.019732970655721927, 0.004452208074546924, -0.03445605604250181, -0.07076319977424485, 0.08981344845492084, -0.03082204942802323, -0.04960869348035414, 0.007478955826168691, -0.010863325536736993, -0.003083199906797788, 0.10709311419736509, -0.10691115797862721, -0.016228654953917093, -0.03080010532469481, 0.054847502392653624, -0.03726677207965316, -0.029266391155138505, -0.02718986721466977, -0.0017478019932951191, 0.014839506985785365, 0.11665883521993853, -0.016467300509130332, -0.006182983269985849, 0.02852655489803301, 0.02198387692467193, 0.058816315495616216, -0.020391520100651064, -0.02194901925727768, -0.06459529577231186, 0.07146290937471365, 0.030013005176420998, 4.452078672183195e-05, -0.010684897343157203, -0.08355925226482504, 0.1023600218932733, -0.08426059231219091, 0.022145676699847273, -0.06112021024817024, 0.15452706861419951, -0.2174066758733319, 1.1450985898980575, -0.18012647938067342, -0.16133737679397625, -0.050119531753255564, 0.11571141796080978], "b": -2.8182981988156106, "auc": 0.7944451467595558, "tasa_base": 0.09436729225726966}
json.dump(coef, open('mask_coef.json', 'w'))
print('AUC del modelo de la máscara:', coef['auc'], '| tasa base de filas puntuadas:', round(coef['tasa_base'], 3))


## Prueba de humo (~3 min)

In [ ]:
!python train_official_gpu_v1.py --train-parquet /kaggle/input/datasets/joaquinfiorina/connectome-train-part1/train_part1.parquet /kaggle/input/datasets/joaquinfiorina/connectome-train-part2/train_part2.parquet --valid-parquet /kaggle/input/datasets/joaquinfiorina/connectome-valid/valid.parquet --train-groups 32 --valid-groups 4 --output-dir /kaggle/tmp/smoke --epochs 1 --cell lstm --hidden 128 --layers 2 --dropout 0.1 --lr 5e-4 --mask-coef mask_coef.json --mask-power 1.0 --seed 1 --model-seed 1 --distill-teachers /kaggle/input/datasets/joaquinfiorina/connectome-models/gru128_distill_seed2029_swa.pt /kaggle/input/datasets/joaquinfiorina/connectome-models/lstm128x2_l1_seed2047_swa.pt /kaggle/input/datasets/joaquinfiorina/connectome-models/lstm128x2_studentLR_seed2072_swa.pt /kaggle/input/datasets/joaquinfiorina/connectome-models/lstm128x2_studentLA_best.pt --distill-cache '' --run-name smoke
import os, json
assert os.path.isfile('/kaggle/tmp/smoke/smoke/metrics.json'), 'La prueba de humo FALLÓ'
print('Prueba de humo OK, valid WP:', json.load(open('/kaggle/tmp/smoke/smoke/metrics.json'))['best_valid_wp'])


## Alumna pesada hacia las filas puntuadas

In [ ]:
!python train_official_gpu_v1.py --train-parquet /kaggle/input/datasets/joaquinfiorina/connectome-train-part1/train_part1.parquet /kaggle/input/datasets/joaquinfiorina/connectome-train-part2/train_part2.parquet --valid-parquet /kaggle/input/datasets/joaquinfiorina/connectome-valid/valid.parquet --train-groups all --valid-groups all --output-dir /kaggle/working/results --save-valid-preds --lr-decay 0.9 --weight-decay 1e-5 --dropout 0.1 --cell lstm --hidden 128 --layers 2 --lr 5e-4 --epochs 10 --swa-start 7 --mask-coef mask_coef.json --mask-power 1.0 --distill-weight 1.0 --distill-cache /kaggle/tmp/soft_S8.npz --distill-teachers /kaggle/input/datasets/joaquinfiorina/connectome-models/gru128_distill_seed2029_swa.pt /kaggle/input/datasets/joaquinfiorina/connectome-models/lstm128x2_l1_seed2047_swa.pt /kaggle/input/datasets/joaquinfiorina/connectome-models/lstm128x2_studentLR_seed2072_swa.pt /kaggle/input/datasets/joaquinfiorina/connectome-models/lstm128x2_studentLA_best.pt --seed 2151 --model-seed 2151 --run-name student_mask || echo 'FALLÓ student_mask'


## Resumen

In [ ]:
import json, os
d = '/kaggle/working/results/student_mask'
m = json.load(open(d + '/metrics.json')) if os.path.isfile(d + '/metrics.json') else None
if m:
    curva = ' / '.join(f"{h['wp']:.4f}" for h in m['history'] if h['tag'] != 'swa')
    print('mejor', round(m['best_valid_wp'], 4), '| SWA', round(m.get('swa_valid_wp', float('nan')), 4))
    print('curva por epoca:', curva)
    print('referencia EXACTA: alumna LR, misma receta sin pesar = 0.6834')
    print(sorted(os.listdir(d)))
else:
    print('FALLO: revisar el error de arriba')
